# Bài 7.2 · Phân loại âm thanh thật

**Câu hỏi của buổi này:** đặc trưng tự tính (MFCC) và embedding của một model đã nghe hai triệu đoạn YouTube (AST) khác nhau bao xa trên cùng một bài, và chia fold sai thì điểm ảo đến mức nào?

**ESC-50** có 2 000 đoạn ghi âm dài 5 giây thuộc 50 loại âm thanh môi trường: chó sủa, mưa, gõ cửa, còi xe… Mỗi loại 40 đoạn, chia sẵn 5 fold; các đoạn cắt từ cùng một bản ghi gốc luôn nằm chung một fold.

| Mục | Sẽ nghe và thấy |
|---|---|
| 1. Dữ liệu | bảng 5 fold, bốn âm thanh mẫu kèm trình phát, waveform và log-mel |
| 2. MFCC và logistic | MFCC của một đoạn, accuracy từng fold |
| 3. AST nghe gì khi chưa học | bảng ba nhãn AudioSet cao nhất của bốn đoạn |
| 4. Embedding AST và logistic | accuracy từng fold, so với MFCC |
| 5. Chia fold sai | điểm ảo khi trộn ngẫu nhiên các đoạn cùng bản ghi |
| 6. Còn nhầm gì | lớp khó nhất, cặp hay nhầm, nghe thử một cặp |
| 7. Tóm tắt | một bảng các con số chính |

Các dòng ✓ là phép kiểm in ra: chạy lại mà thấy ✗ thì có gì đó sai. Bật **GPU** và **Internet** trong *Session options*; chạy hết khoảng 6 phút. Bài giảng: [ai-canuc80k · bài 7.2](https://ai-canuc80k.ai-web-deploy-20261010.workers.dev/#/lesson/7.2).

In [ ]:
# Ô tiện ích: định dạng bảng, dòng kiểm ✓/✗, nhãn số trên biểu đồ. Không cần đọc để theo bài.
import warnings, contextlib, io, logging
warnings.filterwarnings('ignore')
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from IPython.display import display, HTML

plt.rcParams.update({
    'figure.dpi': 110, 'figure.figsize': (8, 4), 'font.size': 11, 'axes.titlesize': 12, 'axes.titleweight': 'bold',
    'axes.spines.top': False, 'axes.spines.right': False, 'axes.grid': True, 'grid.alpha': .25,
    'legend.frameon': False, 'image.cmap': 'viridis',
})
C = {'blue': '#3b6ea8', 'clay': '#c2603e', 'sand': '#b8892b', 'green': '#4d7337', 'gray': '#8a877f'}
ORDER = [C['blue'], C['clay'], C['sand'], C['green'], C['gray']]
plt.rcParams['axes.prop_cycle'] = plt.cycler(color=ORDER)


def table(df, title=None, fmt=None, best=None, lower_is_better=False, index=False):
    """In một bảng gọn: định dạng số, tô đậm ô tốt nhất của cột `best`."""
    st = df.style.format(fmt, na_rep='–')
    if best is not None:
        col = df[best]
        target = col.min() if lower_is_better else col.max()
        st = st.apply(lambda s: ['font-weight:700;background:#f5e6dd' if (s.name == best and v == target) else '' for v in s], axis=0)
    text_cols = [c for c in df.columns if not pd.api.types.is_numeric_dtype(df[c])]
    if text_cols:
        st = st.set_properties(subset=text_cols, **{'text-align': 'left'})
    if not index:
        st = st.hide(axis='index')
    st = st.set_table_styles([
        {'selector': 'th', 'props': 'text-align:left;padding:6px 12px;border-bottom:2px solid #d3cebf;font-weight:600'},
        {'selector': 'td', 'props': 'padding:5px 12px;border-bottom:1px solid #e6e2d6'},
        {'selector': 'caption', 'props': 'caption-side:top;text-align:left;font-weight:700;font-size:14px;padding:4px 0 8px;white-space:nowrap'},
    ])
    if title:
        st = st.set_caption(title)
    display(st)


def check(msg, ok):
    """Dòng kiểm tra in ra: ✓ khi đúng, ✗ khi sai (không dừng notebook)."""
    print(('✓ ' if ok else '✗ ') + msg)


def bar_labels(ax, fmt='{:.2f}'):
    """Ghi giá trị lên đầu mỗi thanh của biểu đồ cột."""
    for cont in ax.containers:
        ax.bar_label(cont, labels=[fmt.format(v) for v in cont.datavalues], padding=3, fontsize=9)


@contextlib.contextmanager
def quiet():
    """Ẩn thanh tiến trình và dòng thông báo khi tải dữ liệu hay trọng số."""
    logging.disable(logging.WARNING)
    try:
        with contextlib.redirect_stdout(io.StringIO()), contextlib.redirect_stderr(io.StringIO()):
            yield
    finally:
        logging.disable(logging.NOTSET)


def note(text):
    """Một dòng nhận xét in đậm nhẹ dưới output, tính từ số thật của lần chạy."""
    display(HTML(f'<div style="border-left:3px solid #c2603e;padding:4px 10px;margin:6px 0;color:#444">{text}</div>'))

In [ ]:
import io, time
import soundfile as sf, librosa, torch, transformers
from IPython.display import Audio
from transformers import AutoFeatureExtractor, ASTForAudioClassification
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.model_selection import StratifiedKFold
transformers.logging.set_verbosity_error()

device = 'cuda' if torch.cuda.is_available() else 'cpu'
SR = 16000
table(pd.DataFrame([('librosa', librosa.__version__), ('transformers', transformers.__version__),
                    ('thiết bị', torch.cuda.get_device_name(0) if device == 'cuda' else 'CPU')],
                   columns=['thành phần', 'phiên bản']), 'Môi trường chạy')

def play(y, label):
    display(HTML(f'<div style="margin:6px 0 2px;font-weight:600">{label}</div>')); display(Audio(y, rate=SR))

## 1. Dữ liệu

Hai tệp parquet trên Hugging Face chứa nhãn lẫn bytes của tệp WAV gốc (44.1 kHz). Mọi đoạn được resample về 16 kHz, đúng tần số AST đã học (bài 7.1).

In [ ]:
base = 'https://huggingface.co/datasets/ashraq/esc50/resolve/main/data/'
files = ['train-00000-of-00002-2f1ab7b824ec751f.parquet', 'train-00001-of-00002-27425e5c1846b494.parquet']
t = time.time()
df = pd.concat([pd.read_parquet(base + f) for f in files], ignore_index=True)

def decode(b):
    x, sr = sf.read(io.BytesIO(b), dtype='float32')
    return librosa.resample(x, orig_sr=sr, target_sr=SR)
waves = [decode(a['bytes']) for a in df['audio']]
df = df.drop(columns='audio')
t_load = time.time() - t
y = df['target'].to_numpy()
name_of = dict(zip(df['target'], df['category']))
folds = df.groupby('fold').agg(**{'số đoạn': ('target', 'size'), 'số loại': ('target', 'nunique'), 'số bản ghi gốc': ('src_file', 'nunique')}).reset_index()
table(folds, f'ESC-50: {len(df)} đoạn, {df["category"].nunique()} loại (tải và resample mất {t_load:.0f} giây)')
shared = df.groupby('src_file')['fold'].nunique().max()
check('mọi đoạn cắt từ cùng một bản ghi nằm chung một fold', shared == 1)
check(f'mỗi đoạn {len(waves[0]) / SR:.1f} giây ở {SR} Hz', abs(len(waves[0]) / SR - 5) < 0.01)

In [ ]:
cats = ['dog', 'rain', 'door_wood_knock', 'church_bells']
fig, axes = plt.subplots(2, 4, figsize=(16, 5.6))
for j, cat in enumerate(cats):
    i = df.index[df['category'] == cat][0]
    axes[0, j].plot(np.arange(len(waves[i])) / SR, waves[i], lw=.4, color=C['blue']); axes[0, j].set_title(cat); axes[0, j].set_xlabel('giây')
    mel = librosa.power_to_db(librosa.feature.melspectrogram(y=waves[i], sr=SR, n_mels=64), ref=np.max)
    axes[1, j].imshow(mel, origin='lower', aspect='auto', cmap='magma', extent=[0, 5, 0, 64]); axes[1, j].grid(False); axes[1, j].set_xlabel('giây')
axes[0, 0].set_ylabel('biên độ'); axes[1, 0].set_ylabel('dải mel')
plt.tight_layout(); plt.show()
for cat in cats: play(waves[df.index[df['category'] == cat][0]], cat)

## 2. Cách 1: MFCC và hồi quy logistic

MFCC nén mỗi frame log-mel thành 40 hệ số mô tả "hình dáng" phổ. Lấy trung bình và độ lệch chuẩn theo thời gian được một vector 80 số cho mỗi đoạn. Cách này bỏ mất thứ tự thời gian nhưng rẻ và dễ hiểu.

Chấm đúng giao thức của bộ dữ liệu: lần lượt để riêng một fold làm test, học trên bốn fold còn lại.

In [ ]:
def mfcc_feat(x):
    m = librosa.feature.mfcc(y=x, sr=SR, n_mfcc=40)
    return np.concatenate([m.mean(1), m.std(1)])

i0 = df.index[df['category'] == 'dog'][0]
M = librosa.feature.mfcc(y=waves[i0], sr=SR, n_mfcc=40)
fig, axes = plt.subplots(1, 2, figsize=(14, 3.4), gridspec_kw={'width_ratios': [2, 1]})
axes[0].imshow(M, origin='lower', aspect='auto', cmap='RdBu_r', extent=[0, 5, 0, 40]); axes[0].grid(False)
axes[0].set(xlabel='giây', ylabel='hệ số MFCC', title=f'MFCC của tiếng chó: 40 × {M.shape[1]} số')
axes[1].plot(M.mean(1), color=C['blue'], label='trung bình theo thời gian'); axes[1].plot(M.std(1), color=C['clay'], label='độ lệch chuẩn')
axes[1].set(xlabel='hệ số', title='80 số đại diện cho cả đoạn'); axes[1].legend()
plt.tight_layout(); plt.show()

t = time.time()
F_mfcc = np.stack([mfcc_feat(x) for x in waves])
t_mfcc = time.time() - t

def five_fold(F, splits=None):
    accs, pred = [], np.zeros(len(y), dtype=int)
    splits = splits or [((df['fold'] != k).to_numpy(), (df['fold'] == k).to_numpy()) for k in sorted(df['fold'].unique())]
    for tr, te in splits:
        clf = make_pipeline(StandardScaler(), LogisticRegression(max_iter=3000, C=0.5)).fit(F[tr], y[tr])
        pred[te] = clf.predict(F[te]); accs.append((pred[te] == y[te]).mean())
    return np.array(accs), pred

acc_mfcc, pred_mfcc = five_fold(F_mfcc)
check(f'MFCC: {F_mfcc.shape[0]} đoạn × {F_mfcc.shape[1]} số ({t_mfcc:.0f} giây)', F_mfcc.shape == (2000, 80))
note(f'MFCC + logistic: accuracy trung bình 5 fold <b>{acc_mfcc.mean():.3f}</b> (các fold từ {acc_mfcc.min():.3f} đến {acc_mfcc.max():.3f}); đoán bừa 50 lớp được 0.02.')

## 3. AST nghe gì khi chưa học gì từ ESC-50

AST (Audio Spectrogram Transformer) cắt log-mel thành các mảnh 16 × 16 rồi đưa qua Transformer, giống ViT với ảnh. Model này đã học trên **AudioSet** (hai triệu đoạn YouTube, 527 nhãn). Cho nó nghe bốn đoạn ở trên:

In [ ]:
name = 'MIT/ast-finetuned-audioset-10-10-0.4593'
with quiet():
    fe = AutoFeatureExtractor.from_pretrained(name)
    ast = ASTForAudioClassification.from_pretrained(name).to(device).eval()
rows = []
for cat in cats:
    i = df.index[df['category'] == cat][0]
    inp = fe(waves[i], sampling_rate=SR, return_tensors='pt').to(device)
    with torch.no_grad():
        p = ast(**inp).logits.sigmoid()[0].cpu()
    top = p.topk(3)
    rows.append([cat] + [f'{ast.config.id2label[int(j)]} ({v:.2f})' for v, j in zip(top.values, top.indices)])
table(pd.DataFrame(rows, columns=['đoạn ESC-50', 'nhãn AudioSet thứ 1', 'thứ 2', 'thứ 3']), 'AST đoán bằng nhãn AudioSet, chưa học gì từ ESC-50')
check(f'đầu vào AST: log-mel {tuple(inp["input_values"].shape[1:])} (frame × mel)', inp['input_values'].shape[-1] == 128)

## 4. Cách 2: embedding của AST và một bộ phân loại tuyến tính

Lấy vector 768 chiều ở cuối Transformer (`pooler_output`) cho mỗi đoạn, giữ nguyên trọng số của AST. Chỉ một hồi quy logistic được học trên các vector này, chấm theo đúng 5 fold như cách 1.

In [ ]:
t = time.time(); embs = []
with torch.no_grad(), torch.autocast(device, dtype=torch.float16, enabled=device == 'cuda'):
    for i in range(0, len(waves), 32):
        inp = fe(waves[i:i + 32], sampling_rate=SR, return_tensors='pt').to(device)
        embs.append(ast.audio_spectrogram_transformer(**inp).pooler_output.float().cpu())
F_ast = torch.cat(embs).numpy(); t_ast = time.time() - t
acc_ast, pred_ast = five_fold(F_ast)

cmp = pd.DataFrame({'fold': list(range(1, 6)) + ['trung bình'], 'MFCC + logistic': list(acc_mfcc) + [acc_mfcc.mean()], 'AST embedding + logistic': list(acc_ast) + [acc_ast.mean()]})
table(cmp, f'Accuracy trên fold để riêng (AST trích embedding mất {t_ast:.0f} giây)', fmt={'MFCC + logistic': '{:.3f}', 'AST embedding + logistic': '{:.3f}'})
fig, ax = plt.subplots(figsize=(8, 3.2))
for k, (n, a, c) in enumerate([('MFCC + logistic', acc_mfcc, C['gray']), ('AST embedding + logistic', acc_ast, C['blue'])]):
    ax.barh(k, a.mean(), color=c, height=.55); ax.scatter(a, np.full(5, k), color='#222', s=18, zorder=3)
    ax.text(a.mean() + 0.01, k + .3, f'{a.mean():.3f}', fontsize=10)
ax.set_yticks([0, 1], ['MFCC + logistic', 'AST embedding + logistic']); ax.set_xlim(0, 1.05); ax.invert_yaxis(); ax.grid(axis='y', visible=False)
ax.set(xlabel='accuracy (thanh: trung bình, chấm: từng fold)', title='Cùng một bộ phân loại tuyến tính, khác đặc trưng')
plt.tight_layout(); plt.show()
check('AST vượt MFCC ở cả 5 fold', (acc_ast > acc_mfcc).all())
note(f'Toàn bộ khác biệt <b>{acc_mfcc.mean():.3f} → {acc_ast.mean():.3f}</b> đến từ những gì AST đã học trên AudioSet; trên ESC-50 chỉ có một lớp tuyến tính được học.')

## 5. Chia fold sai thì sao

Mỗi bản ghi gốc của ESC-50 thường được cắt thành vài đoạn 5 giây, giống nhau về micro, phòng, tiếng nền. Nếu **trộn ngẫu nhiên** 2 000 đoạn rồi chia 5 fold, các đoạn anh em của cùng một bản ghi rơi vào cả train lẫn test: model nhận ra bản ghi chứ không cần nhận ra loại âm.

In [ ]:
rand = [(np.isin(np.arange(len(y)), b, invert=True), np.isin(np.arange(len(y)), b)) for _, b in StratifiedKFold(5, shuffle=True, random_state=0).split(F_mfcc, y)]
leak = pd.DataFrame([(n, five_fold(F)[0].mean(), five_fold(F, rand)[0].mean()) for n, F in [('MFCC + logistic', F_mfcc), ('AST embedding + logistic', F_ast)]],
                    columns=['đặc trưng', 'fold chính thức', 'trộn ngẫu nhiên'])
leak['điểm ảo thêm'] = leak['trộn ngẫu nhiên'] - leak['fold chính thức']
table(leak, 'Accuracy 5 fold: chia đúng và chia sai', fmt={c: '{:.3f}' for c in ['fold chính thức', 'trộn ngẫu nhiên']} | {'điểm ảo thêm': '{:+.3f}'})
multi = (df.groupby('src_file').size() > 1).mean()
note(f"{multi:.0%} bản ghi gốc có từ hai đoạn trở lên. Trộn ngẫu nhiên làm accuracy của MFCC đổi <b>{leak['điểm ảo thêm'].iloc[0]:+.3f}</b>, của AST {leak['điểm ảo thêm'].iloc[1]:+.3f}. "
     f"Phần tăng ấy không đến từ việc nghe giỏi hơn mà từ việc gặp lại đoạn anh em của cùng một bản ghi.")

## 6. Còn nhầm gì

Mỗi đoạn được dự đoán đúng một lần, khi fold của nó làm test (đặc trưng AST).

In [ ]:
per = pd.DataFrame({'loại': [name_of[k] for k in range(50)], 'accuracy': [(pred_ast[y == k] == k).mean() for k in range(50)]}).sort_values('accuracy')
fig, ax = plt.subplots(figsize=(9, 3.6))
w = per.head(10)
ax.barh(w['loại'], w['accuracy'], color=C['clay']); ax.set_xlim(0, 1.08); ax.grid(axis='y', visible=False); bar_labels(ax, '{:.2f}')
ax.set_title('10 loại khó nhất với AST'); plt.tight_layout(); plt.show()
wrong = y != pred_ast
pairs = pd.Series([(name_of[a], name_of[b]) for a, b in zip(y[wrong], pred_ast[wrong])]).value_counts().head(8)
table(pd.DataFrame([(a, b, n) for (a, b), n in pairs.items()], columns=['loại thật', 'AST đoán', 'số đoạn']), f'Sai {wrong.sum()} / {len(y)} đoạn: tám cặp hay nhầm nhất')
(a, b) = pairs.index[0]
ia = int(np.flatnonzero(wrong & (y == df.loc[df['category'] == a, 'target'].iloc[0]) & (pred_ast == df.loc[df['category'] == b, 'target'].iloc[0]))[0])
ib = int(df.index[(df['category'] == b)][0])
play(waves[ia], f'{a} (bị đoán là {b})'); play(waves[ib], f'{b} thật, để so')

## 7. Tóm tắt

In [ ]:
summary = [('MFCC + logistic, accuracy 5 fold', f'{acc_mfcc.mean():.3f}'),
           ('AST embedding + logistic, accuracy 5 fold', f'{acc_ast.mean():.3f}'),
           ('điểm ảo khi trộn ngẫu nhiên: MFCC / AST', f"{leak['điểm ảo thêm'].iloc[0]:+.3f} / {leak['điểm ảo thêm'].iloc[1]:+.3f}"),
           ('loại khó nhất với AST', f"{per['loại'].iloc[0]} ({per['accuracy'].iloc[0]:.2f})"),
           ('cặp nhầm nhiều nhất', f'{a} → {b} ({pairs.iloc[0]} đoạn)')]
table(pd.DataFrame(summary, columns=['điều đo', 'kết quả']), 'Các con số chính của notebook')

**Thử thêm khi dạy:** ghi âm một tiếng động trong lớp (vỗ tay, gõ bàn), tải lên Kaggle rồi cho AST ở mục 3 nghe thử; đổi `n_mfcc=40` thành 13; thêm `delta` của MFCC vào vector đặc trưng.